<a href="https://colab.research.google.com/github/george-hawkins/japanese/blob/master/fbcnn_jpeg_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Connect to Google Drive
from google.colab import auth
from google.colab import drive

# Triggers authentication flow.
auth.authenticate_user()

# And mount drive.
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# @title Specify paths
from pathlib import Path

root = "/content/drive/MyDrive/colab/fbcnn"
input_dir = Path(f"{root}/input")
output_dir = Path(f"{root}/output")

In [4]:
# @title Determine input images
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}

image_filenames = [path for path in sorted(input_dir.glob("*")) if path.suffix.lower() in IMAGE_SUFFIXES]

print(f"{len(image_filenames)} images found in {input_dir}")

1 images found in /content/drive/MyDrive/colab/fbcnn/input


In [5]:
# @title Check images are not color
from PIL import Image

for image_filename in image_filenames:
  mode = Image.open(image_filename).mode
  is_color = mode not in ("L", "1")
  if is_color:
    # This notebook is setup for grayscale images.
    # For color, you need to change the `.pth` model and use `main_test_fbcnn_color.py` later.
    raise RuntimeError(f"{image_filename} may be grayscale but it has been saved as a color JPEG")

print("all images were saved as grayscale JPEG")

all images were saved as grayscale JPEG


In [25]:
# @title Install FBCNN, dependencies and model
!curl -O https://raw.githubusercontent.com/jiaxi-jiang/FBCNN/refs/heads/main/models/network_fbcnn.py

!pip install torch torchvision numpy opencv-python

if not Path("main_test_fbcnn_gray.py").exists():
  !curl -L -O https://github.com/jiaxi-jiang/FBCNN/releases/download/v1.0/fbcnn_gray.pth

print("FBCNN, model and dependencies successfully installed")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 14063  100 14063    0     0  69765      0 --:--:-- --:--:-- --:--:-- 69965
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  274M  100  274M    0     0   178M      0  0:00:01  0:00:01 --:--:--  273M
FBCNN, model and dependencies successfully installed


In [27]:
# @title Check CUDA
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available")

In [28]:
# @title Python restore_jpeg function
# Largely generated by Claude using FBCNN/main_test_fbcnn_gray.py as a basis
from pathlib import Path
import cv2
import torch
import time

# May 16, 2026 - an odd linter bug marks network_fbcnn as unresolvable even when it is. Hence the `ignore`.
from network_fbcnn import FBCNN  # type: ignore

def restore_jpeg(
    input_path,
    output_path,
    *,
    model_path="fbcnn_gray.pth",
    quality_factor=None,
):
    """Restore a JPEG-compressed grayscale image using FBCNN.

    Args:
        input_path: Path to the JPEG-compressed input image. Color images are
            converted to grayscale on read.
        output_path: Where to write the restored PNG.
        model_path: Path to FBCNN grayscale weights (.pth).
        quality_factor: Optional QF in [1, 100] to condition the model on.
            If None, the model uses its own predicted QF.

    Returns:
        The model's predicted quality factor as an int in [0, 100].
    """
    t0 = time.perf_counter()

    input_path = Path(input_path)
    output_path = Path(output_path)
    model_path = Path(model_path)

    device = torch.device("cuda")

    img = cv2.imread(str(input_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"could not read image: {input_path}")

    img_tensor = (
        torch.from_numpy(img).float().div(255.0)
        .unsqueeze(0).unsqueeze(0).to(device)
    )

    model = FBCNN(in_nc=1, out_nc=1, nc=[64, 128, 256, 512], nb=4, act_mode="R")
    model.load_state_dict(torch.load(str(model_path), map_location=device))
    model.eval().to(device)

    print(f"loaded model from {model_path}")
    print(f"input: {input_path} ({img.shape[1]}x{img.shape[0]}, grayscale)")

    # Clear setup so we can gauge VRAM usage.
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():
        if quality_factor is None:
            print("conditioning: model picks its own QF")
            restored, qf_raw = model(img_tensor)
        else:
            if not 1 <= quality_factor <= 100:
                raise ValueError(
                    f"quality_factor must be in [1, 100], got {quality_factor}"
                )
            qf_input = torch.tensor([[1.0 - quality_factor / 100.0]], device=device)
            print(f"conditioning: user-supplied QF = {quality_factor}")
            restored, qf_raw = model(img_tensor, qf_input)

    # Output VRAM usage.
    peak_gb = torch.cuda.max_memory_reserved() / 1024**3
    print(f"Peak VRAM {peak_gb:.2f} GiB")
    elapsed = time.perf_counter() - t0
    print(f"Elapsed time: {elapsed:.2f}s")

    predicted_qf = round(float((1.0 - qf_raw) * 100.0))
    print(f"model-predicted QF: {predicted_qf}")

    restored_np = (
        restored.squeeze().clamp(0, 1).mul(255.0).round().byte().cpu().numpy()
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(output_path), restored_np):
        raise IOError(f"could not write image: {output_path}")
    print(f"saved restored image to {output_path}")

    return predicted_qf

In [32]:
qf_values = []
#qf_values = [45, 50, 55, 60, 65, 70]

def process_images(qf=None):
  for image_filename in image_filenames:
    output_filename = image_filename.stem
    if qf is not None:
      output_filename = f"{output_filename}-{qf}"
    output_filename = output_dir / f"{output_filename}.png"
    restore_jpeg(image_filename, output_filename, quality_factor=qf)

if qf_values is None or len(qf_values) == 0:
    process_images()
else:
  for qf in qf_values:
    process_images(qf)

loaded model from fbcnn_gray.pth
input: /content/drive/MyDrive/colab/fbcnn/input/crop.png (800x800, grayscale)
conditioning: model picks its own QF
Peak VRAM 1.08 GiB
Elapsed time: 1.36s
model-predicted QF: 50
saved restored image to /content/drive/MyDrive/colab/fbcnn/output/crop.png
